# Arabic Long-Form ASR Improvement in Modern Standard Arabic

**Track:** Group 2 - Long-Form Transcription in Arabic (MSA)  
**Domain:** Arabic media, education, broadcasting, lectures, interviews, and archived content  
**Goal:** Improve transcription quality for long Arabic audio using Whisper adaptation techniques.

This unified notebook merges the project code into one clear pipeline:

1. Define the real-world problem.
2. Load Arabic ASR data and manifests.
3. Apply Arabic normalization and filtering.
4. Evaluate ASR using WER and CER.
5. Run long-form chunking experiments.
6. Prepare Whisper fine-tuning experiments with different dataset sizes.
7. Apply data augmentation: noise and speed perturbation.
8. Compare results and produce presentation-ready findings.

## 1. Problem Statement

Arabic ASR systems often underperform compared to English ASR systems because Arabic has:

- Limited high-quality labeled speech data.
- Rich morphology and many word forms.
- Dialect variation.
- Diacritics and spelling variants.
- Long-form audio issues such as topic changes, pauses, repeated phrases, and context loss.

**Project objective:** improve transcription quality for Modern Standard Arabic using adaptation techniques such as preprocessing, Arabic normalization, dataset-size experiments, data augmentation, and Whisper fine-tuning.

## 2. System Pipeline

```text
Long Arabic Audio
        ↓
Audio Preprocessing
16 kHz mono conversion, filtering, chunking
        ↓
Whisper ASR Baseline
        ↓
Arabic Text Normalization
        ↓
WER / CER Evaluation
        ↓
Adaptation Experiments
dataset size + augmentation + fine-tuning
        ↓
Improved Whisper Model
        ↓
Final Transcript + Results Table
```

## 3. Setup

Run this cell in Colab or a local virtual environment.

Important: `google/fleurs` currently depends on a Hugging Face dataset loading script. Newer `datasets>=4` versions removed support for those scripts, so this project pins `datasets<4`.

After installing in Colab/Kaggle, restart the runtime/session before loading FLEURS.

In [19]:
# Uncomment when running in Colab or a fresh environment.
#!pip uninstall -y datasets huggingface_hub
#!pip install -q --no-cache-dir "datasets==3.6.0" "huggingface_hub<1.0.0" transformers accelerate evaluate jiwer soundfile librosa audiomentations

In [20]:
from __future__ import annotations

import csv
import json
import math
import os
import random
import re
import shutil
import subprocess
import tempfile
from dataclasses import dataclass
from pathlib import Path

import numpy as np

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)

Project root: /kaggle/working


## 4. Arabic Normalization

Normalization is important because two Arabic transcripts can be semantically similar but written differently.

This project normalizes:

- Diacritics.
- Tatweel.
- Alef variants.
- Ya / Alef Maqsura variants.
- Hamza variants.
- Arabic-Indic digits.
- Punctuation and extra spaces.

In [21]:
AR_DIACRITICS_RE = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")
PUNCT_RE = re.compile(r"[^\w\s\u0600-\u06FF]", re.UNICODE)
SPACE_RE = re.compile(r"\s+")


def normalize_arabic(text: str) -> str:
    text = text.strip()
    text = AR_DIACRITICS_RE.sub("", text)
    text = text.replace("ـ", "")
    text = re.sub("[إأآٱ]", "ا", text)
    text = text.replace("ى", "ي")
    text = text.replace("ؤ", "و")
    text = text.replace("ئ", "ي")
    text = text.replace("ة", "ه")
    text = text.translate(str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789"))
    text = PUNCT_RE.sub(" ", text)
    text = SPACE_RE.sub(" ", text)
    return text.strip()


examples = [
    "إِنَّ اللُّغَةَ العربيّةَ مهمّةٌ.",
    "آثارُ التَّطويل ـــــ والهمزات تؤثّر في التقييم.",
]

for item in examples:
    print("Original:  ", item)
    print("Normalized:", normalize_arabic(item))
    print()

Original:   إِنَّ اللُّغَةَ العربيّةَ مهمّةٌ.
Normalized: ان اللغه العربيه مهمه

Original:   آثارُ التَّطويل ـــــ والهمزات تؤثّر في التقييم.
Normalized: اثار التطويل والهمزات توثر في التقييم



## 5. Evaluation Metrics: WER and CER

**WER** measures word-level transcription errors.  
**CER** measures character-level errors, which is useful for Arabic spelling variation.

Lower values are better.

In [22]:
def edit_distance(a, b) -> int:
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, start=1):
        cur = [i]
        for j, cb in enumerate(b, start=1):
            cur.append(min(
                prev[j] + 1,
                cur[j - 1] + 1,
                prev[j - 1] + (ca != cb),
            ))
        prev = cur
    return prev[-1]


def wer(reference: str, hypothesis: str) -> float:
    ref_words = reference.split()
    hyp_words = hypothesis.split()
    if not ref_words:
        return 0.0 if not hyp_words else 1.0
    return edit_distance(ref_words, hyp_words) / len(ref_words)


def cer(reference: str, hypothesis: str) -> float:
    ref_chars = reference.replace(" ", "")
    hyp_chars = hypothesis.replace(" ", "")
    if not ref_chars:
        return 0.0 if not hyp_chars else 1.0
    return edit_distance(ref_chars, hyp_chars) / len(ref_chars)


reference = "هذا اختبار قصير لنظام التعرف على الكلام"
hypothesis = "هذا اختبار لنظام التعرف علي الكلام"

print("Raw WER:", round(wer(reference, hypothesis), 3))
print("Normalized WER:", round(wer(normalize_arabic(reference), normalize_arabic(hypothesis)), 3))
print("Normalized CER:", round(cer(normalize_arabic(reference), normalize_arabic(hypothesis)), 3))

Raw WER: 0.286
Normalized WER: 0.143
Normalized CER: 0.121


## 6. Manifest Format

The project uses JSONL manifests. Each row represents one audio sample.

```json
{"id": "lecture_001", "audio": "data/audio/lecture_001.wav", "reference": "gold Arabic transcript"}
```

In [23]:
def read_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def write_jsonl(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


demo_manifest = DATA_DIR / "demo.jsonl"
if demo_manifest.exists():
    rows = read_jsonl(demo_manifest)
    print("Demo manifest rows:", len(rows))
    print(rows[0].keys())
    print("Reference preview:", rows[0]["reference"][:180], "...")
else:
    print("No demo manifest found.")

No demo manifest found.


## 7. Audio Preprocessing and Long-Form Chunking

Long audio is handled by:

1. Converting audio to 16 kHz mono WAV.
2. Splitting it into fixed-size chunks.
3. Using optional overlap between chunks.
4. Sending each chunk to Whisper.
5. Merging chunk transcripts.
6. Removing repeated overlap text.

In [24]:
def run_command(cmd: list[str]) -> str:
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(result.stderr)
    return result.stdout.strip()


def audio_duration_seconds(path: Path) -> float:
    out = run_command([
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        str(path),
    ])
    return float(out)


def extract_wav(input_path: Path, wav_path: Path, start: float | None = None, duration: float | None = None) -> None:
    cmd = ["ffmpeg", "-y"]
    if start is not None:
        cmd.extend(["-ss", f"{start:.3f}"])
    cmd.extend(["-i", str(input_path)])
    if duration is not None:
        cmd.extend(["-t", f"{duration:.3f}"])
    cmd.extend(["-ar", "16000", "-ac", "1", "-c:a", "pcm_s16le", str(wav_path), "-loglevel", "error"])
    run_command(cmd)


def deduplicate_overlap(prev_text: str, new_text: str, max_overlap_words: int = 18) -> str:
    prev_words = prev_text.split()
    new_words = new_text.split()
    max_k = min(max_overlap_words, len(prev_words), len(new_words))
    for k in range(max_k, 0, -1):
        if prev_words[-k:] == new_words[:k]:
            return " ".join(new_words[k:])
    return new_text

## 8. Whisper.cpp Long-Form Transcription Function

This cell is for local transcription using `whisper.cpp`.

Requirements:

- `ffmpeg`
- `ffprobe`
- `whisper-cli`
- Whisper model file, for example `ggml-base.bin`

In [25]:
@dataclass
class WhisperCppConfig:
    whisper_bin: str = "whisper-cli"
    model: Path = Path("ggml-base.bin")
    language: str = "ar"
    threads: int = 8
    chunk_seconds: float = 60.0
    overlap_seconds: float = 2.0
    context_words: int = 24
    use_prompt_context: bool = False


def transcript_context(text: str, max_words: int) -> str:
    words = text.split()
    if max_words <= 0 or not words:
        return ""
    return " ".join(words[-max_words:])


print("Whisper.cpp config class is ready.")
print("Use asr_experiments.py for full transcription runs, or copy these settings into CLI commands.")

Whisper.cpp config class is ready.
Use asr_experiments.py for full transcription runs, or copy these settings into CLI commands.


## 9. Command-Line Experiments

These commands run the main long-form experiments from the merged script.

Best completed setting from this repository:

- 60-second chunks.
- 2-second overlap.
- No prompt context.

In [26]:
# Example commands. Update --model to your local Whisper model path before running.

baseline_command = '''
python asr_experiments.py transcribe \
  --manifest data/longform_30min.jsonl \
  --model /path/to/ggml-base.bin \
  --system-name whisper_base_30min_no_context \
  --chunk-seconds 60 \
  --overlap-seconds 2 \
  --no-prompt-context \
  --out results/whisper_base_30min_no_context_predictions.jsonl
'''

evaluate_command = '''
python asr_experiments.py evaluate \
  --pred results/whisper_base_30min_no_context_predictions.jsonl \
  --out results/whisper_base_30min_no_context_metrics.json \
  --csv results/whisper_base_30min_no_context_metrics.csv
'''

print(baseline_command)
print(evaluate_command)


python asr_experiments.py transcribe   --manifest data/longform_30min.jsonl   --model /path/to/ggml-base.bin   --system-name whisper_base_30min_no_context   --chunk-seconds 60   --overlap-seconds 2   --no-prompt-context   --out results/whisper_base_30min_no_context_predictions.jsonl


python asr_experiments.py evaluate   --pred results/whisper_base_30min_no_context_predictions.jsonl   --out results/whisper_base_30min_no_context_metrics.json   --csv results/whisper_base_30min_no_context_metrics.csv



## 10. Load Existing Experiment Results

The repository already includes completed metrics. This section reads them and creates a comparison table.

In [27]:
def load_metrics_csv(path: Path) -> dict:
    with path.open(encoding="utf-8") as f:
        reader = csv.DictReader(f)
        rows = list(reader)
    if not rows:
        return {}
    row = rows[0]
    return {
        "id": row["id"],
        "system": row["system"],
        "wer_raw": float(row["wer_raw"]),
        "wer_normalized": float(row["wer_normalized"]),
        "cer_normalized": float(row["cer_normalized"]),
    }


metric_files = sorted(RESULTS_DIR.glob("*metrics.csv"))
metrics = [load_metrics_csv(path) for path in metric_files]
metrics = [m for m in metrics if m]

print(f"Loaded {len(metrics)} metric files.")
for m in metrics:
    print(f"{m['system']:<45} WER(norm)={m['wer_normalized']:.4f} CER(norm)={m['cer_normalized']:.4f}")

Loaded 0 metric files.


In [28]:
try:
    import pandas as pd

    df = pd.DataFrame(metrics)
    display(df.sort_values("wer_normalized"))
except Exception:
    sorted_metrics = sorted(metrics, key=lambda x: x["wer_normalized"])
    for row in sorted_metrics:
        print(row)

## 11. Results Visualization

This chart compares normalized WER and CER across available systems.

In [29]:
try:
    import matplotlib.pyplot as plt
    import pandas as pd

    df = pd.DataFrame(metrics).sort_values("wer_normalized")
    labels = df["system"].str.replace("whisper_", "", regex=False)

    plt.figure(figsize=(12, 5))
    plt.bar(labels, df["wer_normalized"], label="Normalized WER")
    plt.bar(labels, df["cer_normalized"], label="Normalized CER", alpha=0.75)
    plt.xticks(rotation=35, ha="right")
    plt.ylabel("Error rate")
    plt.title("Arabic ASR Evaluation Results")
    plt.legend()
    plt.tight_layout()
    plt.show()
except Exception as exc:
    print("Chart skipped:", exc)

Chart skipped: 'wer_normalized'


## 12. Fine-Tuning Whisper on Arabic Data

This section is designed for Colab or Kaggle GPU.

Experiments:

- **E4:** 50 training samples, no augmentation.
- **E5:** 150 training samples, no augmentation.
- **E6:** 150 training samples, with augmentation.

The repository results show that increasing the dataset size improved WER/CER, while the tested augmentation setup did not improve over E5.

In [30]:
# GPU fine-tuning setup. Run in Colab/Kaggle, not required for local results inspection.

MODEL_NAME = "openai/whisper-small"
LANGUAGE = "Arabic"
TASK = "transcribe"

SMALL_TRAIN_SIZE = 50
LARGE_TRAIN_SIZE = 150
EVAL_SIZE = 20

print("Fine-tuning configuration:")
print("Model:", MODEL_NAME)
print("Small train size:", SMALL_TRAIN_SIZE)
print("Large train size:", LARGE_TRAIN_SIZE)
print("Eval size:", EVAL_SIZE)

Fine-tuning configuration:
Model: openai/whisper-small
Small train size: 50
Large train size: 150
Eval size: 20


In [31]:
# Dataset loading template for FLEURS Arabic Egypt.
# Uncomment in Colab/Kaggle after installing the pinned dependencies above
# and restarting the runtime/session.

from datasets import load_dataset
import datasets

print("datasets version:", datasets.__version__)
if int(datasets.__version__.split(".")[0]) >= 4:
    raise RuntimeError(
        "google/fleurs uses a dataset loading script. "
        "Run: pip install 'datasets<4.0.0' 'huggingface_hub<1.0.0', then restart the runtime."
    )

dataset = load_dataset("google/fleurs", "ar_eg", trust_remote_code=True)
train_small = dataset["train"].shuffle(seed=42).select(range(SMALL_TRAIN_SIZE))
train_large = dataset["train"].shuffle(seed=42).select(range(LARGE_TRAIN_SIZE))
eval_set = dataset["validation"].shuffle(seed=42).select(range(EVAL_SIZE))

print(dataset)

datasets version: 3.6.0
DatasetDict({
    train: Dataset({
        features: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id'],
        num_rows: 2104
    })
    validation: Dataset({
        features: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id'],
        num_rows: 295
    })
    test: Dataset({
        features: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id'],
        num_rows: 428
    })
})


### Kaggle/Colab FLEURS Loading Fix

If you see:

```text
RuntimeError: Dataset scripts are no longer supported, but found fleurs.py
```

it means the runtime is still using `datasets>=4`. Run the next install cell, restart the session, then run the load cell after it.

In [32]:
# Run this cell once, then restart the Kaggle/Colab session before continuing.
# Do not run load_dataset until after the restart.

# !pip uninstall -y datasets huggingface_hub
# !pip install -q --no-cache-dir "datasets==3.6.0" "huggingface_hub<1.0.0" transformers accelerate evaluate jiwer soundfile librosa audiomentations

In [33]:
# Run this only after restarting the runtime/session.

import datasets
from datasets import load_dataset

print("datasets version:", datasets.__version__)
assert int(datasets.__version__.split(".")[0]) < 4, "Restart did not apply. datasets must be <4."

dataset = load_dataset("google/fleurs", "ar_eg", trust_remote_code=True)
train_small = dataset["train"].shuffle(seed=42).select(range(SMALL_TRAIN_SIZE))
train_large = dataset["train"].shuffle(seed=42).select(range(LARGE_TRAIN_SIZE))
eval_set = dataset["validation"].shuffle(seed=42).select(range(EVAL_SIZE))

print(dataset)

datasets version: 3.6.0
DatasetDict({
    train: Dataset({
        features: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id'],
        num_rows: 2104
    })
    validation: Dataset({
        features: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id'],
        num_rows: 295
    })
    test: Dataset({
        features: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id'],
        num_rows: 428
    })
})


## 13. Data Augmentation

Two simple augmentation strategies:

- **Noise injection:** improves robustness to real-world background noise.
- **Speed perturbation:** simulates different speaking speeds.

In [34]:
def add_noise(audio: np.ndarray, noise_factor: float = 0.005) -> np.ndarray:
    noise = np.random.randn(len(audio)).astype(np.float32)
    augmented = audio + noise_factor * noise
    return augmented.astype(np.float32)


def speed_perturb_placeholder(audio: np.ndarray, speed: float = 1.05) -> np.ndarray:
    # In the full GPU notebook, use librosa.effects.time_stretch for real speed perturbation.
    # This placeholder keeps the unified notebook lightweight and dependency-safe.
    return audio


dummy_audio = np.zeros(16000, dtype=np.float32)
print("Noise augmentation output shape:", add_noise(dummy_audio).shape)
print("Speed perturbation placeholder shape:", speed_perturb_placeholder(dummy_audio).shape)

Noise augmentation output shape: (16000,)
Speed perturbation placeholder shape: (16000,)


In [35]:
# Full augmentation version for Colab/Kaggle:
#
import librosa

def speed_perturb(audio: np.ndarray, sr: int, speed: float = 1.05) -> np.ndarray:
    return librosa.effects.time_stretch(audio.astype(np.float32), rate=speed)

def augment_example(batch):
    audio = batch["audio"]["array"].astype(np.float32)
    sr = batch["audio"]["sampling_rate"]
    if random.random() < 0.5:
        audio = add_noise(audio)
    if random.random() < 0.5:
        audio = speed_perturb(audio, sr, speed=random.choice([0.95, 1.05]))
    batch["audio"]["array"] = audio
    return batch

## 14. Fine-Tuning Skeleton

This cell shows the structure of the training loop. Run it on GPU after enabling the dataset cells.

In [41]:
  import torch
  from dataclasses import dataclass
  from typing import Any
  from datasets import Audio

  train_small = train_small.cast_column("audio", Audio(sampling_rate=16000))
  eval_set = eval_set.cast_column("audio", Audio(sampling_rate=16000))

  def prepare_example(batch):
      audio = batch["audio"]

      input_features = processor.feature_extractor(
          audio["array"],
          sampling_rate=audio["sampling_rate"],
      ).input_features[0]

      labels = processor.tokenizer(batch["transcription"]).input_ids

      return {
          "input_features": input_features,
          "labels": labels,
      }

  train_small_prepared = train_small.map(
      prepare_example,
      remove_columns=train_small.column_names,
      num_proc=1,
  )

  eval_prepared = eval_set.map(
      prepare_example,
      remove_columns=eval_set.column_names,
      num_proc=1,
  )

  @dataclass
  class DataCollatorSpeechSeq2SeqWithPadding:
      processor: Any

      def __call__(self, features):
          input_features = [
              {"input_features": feature["input_features"]}
              for feature in features
          ]

          batch = self.processor.feature_extractor.pad(
              input_features,
              return_tensors="pt",
          )

          label_features = [
              {"input_ids": feature["labels"]}
              for feature in features
          ]

          labels_batch = self.processor.tokenizer.pad(
              label_features,
              return_tensors="pt",
          )

          labels = labels_batch["input_ids"].masked_fill(
              labels_batch.attention_mask.ne(1),
              -100,
          )

          batch["labels"] = labels
          return batch

  data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

  print(train_small_prepared)
  print(eval_prepared)
  print("Collator ready")

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Dataset({
    features: ['input_features', 'labels'],
    num_rows: 50
})
Dataset({
    features: ['input_features', 'labels'],
    num_rows: 20
})
Collator ready


In [42]:
  trainer = Seq2SeqTrainer(
      model=model,
      args=training_args,
      train_dataset=train_small_prepared,
      eval_dataset=eval_prepared,
      data_collator=data_collator,
      processing_class=processor.feature_extractor,
  )

  trainer.train()

You're using a WhisperTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss,Validation Loss
25,1.121500,0.773604
50,0.258900,0.680568
75,0.191000,0.643638
100,0.156200,0.629085


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 26161, 26435, 28279, 29464, 31650, 32302, 32470, 36865, 42863, 47425, 49870, 50254, 50258, 50360, 50361, 50362], 'begin_suppress_tokens': [220, 50257]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension

TrainOutput(global_step=100, training_loss=0.4956398904323578, metrics={'train_runtime': 516.1321, 'train_samples_per_second': 3.1, 'train_steps_per_second': 0.194, 'total_flos': 3.607317504e+17, 'train_loss': 0.4956398904323578, 'epoch': 25.0})

In [44]:
trainer.save_model("./whisper-arabic-msa-final")
processor.save_pretrained("./whisper-arabic-msa-final")

[]

## 15. Fine-Tuning Results Summary

These are the completed fine-tuning experiments documented in the project files.

In [48]:
fine_tuning_results = [
    {"experiment": "E4 Small fine-tune", "train_samples": 50, "augmentation": "No", "eval_loss": 2.0174, "raw_wer": 0.2899, "normalized_wer": 0.2671, "normalized_cer": 0.0675},
    {"experiment": "E5 Larger fine-tune", "train_samples": 150, "augmentation": "No", "eval_loss": 1.6712, "raw_wer": 0.2769, "normalized_wer": 0.2508, "normalized_cer": 0.0618},
    {"experiment": "E6 Larger fine-tune + augmentation", "train_samples": 150, "augmentation": "Yes", "eval_loss": 1.6823, "raw_wer": 0.2769, "normalized_wer": 0.2508, "normalized_cer": 0.0618},
]

try:
    import pandas as pd
    display(pd.DataFrame(fine_tuning_results))
except Exception:
    for row in fine_tuning_results:
        print(row)

,experiment,train_samples,augmentation,eval_loss,raw_wer,normalized_wer,normalized_cer
0,E4 Small fine-tune,50,No,2.0174,0.2899,0.2671,0.0675
1,E5 Larger fine-tune,150,No,1.6712,0.2769,0.2508,0.0618
2,E6 Larger fine-tune + augmentation,150,Yes,1.6823,0.2769,0.2508,0.0618


## 16. Final Findings

- Long-form Arabic transcription works better when audio is split into manageable chunks.
- In the completed 30-minute benchmark, 60-second chunks performed better than 30-second chunks.
- Prompt context did not improve results in this experiment, likely because it propagated previous recognition errors.
- Arabic normalization reduced measured WER and gave a fairer evaluation.
- Fine-tuning with more Arabic samples improved WER and CER.
- The tested augmentation setup did not improve over the larger non-augmented fine-tune, but it remains useful to test with more realistic noise profiles.

## 17. Presentation/Demo Plan

1. Explain the Arabic ASR problem.
2. Show the pipeline diagram.
3. Show preprocessing and Arabic normalization examples.
4. Show long-form chunking strategy.
5. Show baseline WER/CER.
6. Show fine-tuning experiments E4, E5, and E6.
7. Explain why E5 is the best model-level result.
8. Play a short Arabic audio sample if available.
9. Show reference vs hypothesis transcript.
10. End with limitations and future work.

## 18. References

Radford, A., Kim, J. W., Xu, T., Brockman, G., McLeavey, C., & Sutskever, I. (2022). *Robust speech recognition via large-scale weak supervision*. OpenAI. https://cdn.openai.com/papers/whisper.pdf

Hugging Face. (2026). *Transformers documentation*. https://huggingface.co/docs/transformers/

Hugging Face. (2026). *Datasets documentation*. https://huggingface.co/docs/datasets/

Google Research. (2026). *FLEURS dataset*. https://huggingface.co/datasets/google/fleurs

OpenAI. (2026). *ChatGPT* [Large language model]. https://chat.openai.com